In [ ]:
using Pkg
Pkg.add("Optuna")
Pkg.add("CondaPkg")
CondaPkg.add("optuna")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed Pidfile ──────── v1.3.0
   Installed pixi_jll ─────── v0.63.2+0
   Installed micromamba_jll ─ v2.3.1+0
   Installed UnsafePointers ─ v1.0.0
   Installed MicroMamba ───── v0.1.15
   Installed Optuna ───────── v0.2.1
   Installed CondaPkg ─────── v0.2.33
   Installed PythonCall ───── v0.9.34
    Updating `~/.julia/environments/v1.12/Project.toml`
  [a5d0552b] + Optuna v0.2.1
    Updating `~/.julia/environments/v1.12/Manifest.toml`
⌃ [992eb4ea] + CondaPkg v0.2.33
  [0b3b1443] + MicroMamba v0.1.15
  [a5d0552b] + Optuna v0.2.1
  [fa939f87] + Pidfile v1.3.0
  [6099a3de] + PythonCall v0.9.34
  [e17b2a0c] + UnsafePointers v1.0.0
  [f8abcde7] + micromamba_jll v2.3.1+0
  [4d7b5844] + pixi_jll v0.63.2+0
        Info Packages marked with ⌃ have new versions available and may be upgradable.
Precompiling packages...
  12701.3 ms  ✓ UnsafePointers
   7011.0 ms  ✓ Pidfile
   6749.8 ms  ✓ micromamb

In [ ]:
using MLJ
using ScikitLearn
using DataFrames
using CSV
using Optuna
using StatsBase
using CategoricalArrays
using Impute
using DataVoyager

In [ ]:
train_features = CSV.read("data/training_set_features.csv", DataFrame)
train_labels = CSV.read("data/training_set_labels.csv", DataFrame)

In [ ]:
test = CSV.read("data/test_set_features.csv", DataFrame)

In [ ]:
names(train_features)

## The General Causal Assumption is that:

- behavioral_outside_home -> behavioral_large_gatherings -> behavioral_face_mask -> behavioral_wash_hands -> behavioral_touch_face -> chronic_med_condition

- hhs_geo_region -> behavioral_large_gatherings -> behavioral_touch_face -> chronic_med_condition (can be a genuine causation or bilateral causation)

-  employment_occupation -> income_poverty -> doctor_recc_h1h1

in africa, majority of these happen due to the fact that males are often the ones that find jobs and it was an economic phenomenon that was discovered by economists. For the 2nd causal assumption, it depends on the regions and its traditions of the spread of the h1h1 virus that leads to chronic_mde_condition.

In [ ]:
train = leftjoin(train_features, train_labels, on = :respondent_id)

## EDA

In [ ]:
StatsBase.summarystats(train[:, :h1n1_knowledge])

In [ ]:
for col in names(train)
    println("Column: ", col)

    if eltype(train[!, col]) <: Number
        println(StatsBase.summarystats(train[!, col]))
        println()
    end
end

In [ ]:
#Might have to label encode
ordinal_dict = Dict(
    :age_group => ["18 - 34 Years", "35 - 44 Years", "45 - 54 Years", "55 - 64 Years", "65+ Years"],
    :education => ["< 12 Years", "12 Years", "College Graduate", "Some College", "Post Graduate"],
    :race => ["White", "Black", "Other or Multiple", "Hispanic"],
    :sex => ["Male", "Female"],
    :income_poverty => ["Below Poverty" ,"<= \$75,000, Above Poverty", "> \$75,000"],
    :marital_status => ["Married", "Not Married"],
    :rent_or_own => ["Own", "Rent"],
    :employment_status => [ "Not in Labor Force", "Employed", "Unemployed"],
    :hhs_geo_region	=> ["oxchjgsf", "bhuqouqj", "qufhixun", "lrircsnp", "atmpeygn", "lzgpxyit", "fpwskwrf", "mlyzmhmf", "dqpwygqj", "kbazzjca"],
    :census_msa => ["Non-MSA", "MSA, Not Principle  City", "MSA, Principle City"],
    :employment_industry => ["pxcmvdjn", "rucpziij", "wxleyezf", "saaquncn", "xicduogh", "ldnlellj", "wlfvacwt", "nduyfdeo", "fcxhlnwr", "vjjrobsf", "arjwrbjb", "atmlpfrs", "msuufmds", "xqicxuve", "phxvnwax", "dotnnunm", "mfikgejo", "cfqqtusy", "mcubkhph", "haxffmxo", "qnlwzans"],
    :employment_occupation => ["xgwztkwe", "xtkaffoo", "emcorrxb", "vlluhbov", "xqwwgdyp", "ccgxvspp", "qxajmpny", "kldqjyjy", "mxkfnird", "hfxkjkmi", "bxpfxfdn", "ukymxvdu", "cmhcxjea", "haliazsg", "dlvbwzss", "xzmlyyjv", "oijqvulv", "rcertsgn", "tfqavkke", "hodpvpew", "uqqtjvyb", "pvmttkik", "dcjcmpih"]
)

for (col, levels) in ordinal_dict
    train[!, col] = categorical(train[!, col], ordered=true, levels=levels)
    train[!, Symbol(col, "_enc")] = levelcode.(train[!, col])
end

for (col, levels) in ordinal_dict
    test[!, col] = categorical(test[!, col], ordered=true, levels=levels)
    test[!, Symbol(col, "_enc")] = levelcode.(test[!, col])
end

In [ ]:
unique(train[:, :employment_occupation])

In [ ]:
test

In [ ]:
#Dropping columns so that only encoded and imputted values remain
train = select(train, Not([:age_group, :education, :race, :sex, :income_poverty, :marital_status, :rent_or_own, :employment_status, :hhs_geo_region, :census_msa, :employment_industry, :employment_occupation]))
test = select(test, Not([:age_group, :education, :race, :sex, :income_poverty, :marital_status, :rent_or_own, :employment_status, :hhs_geo_region, :census_msa, :employment_industry, :employment_occupation]))

train = Impute.substitute(train; statistic=mode())
test = Impute.substitute(test; statistic=mode())

## Hyperparameter Tuning